<a href="https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Install packages (only needed once per Colab session)
!pip -q install duckdb huggingface_hub

import duckdb
from huggingface_hub import snapshot_download
from google.colab import userdata

# Read your Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Download (or reuse cached) warehouse
repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
)

# Connect DuckDB
con = duckdb.connect()

# Create views
con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM read_parquet('{repo_path}/fact_content_daily_performance/month=2026-03/*.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT *
FROM read_parquet('{repo_path}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW dim_clients AS
SELECT *
FROM read_parquet('{repo_path}/dim_clients.parquet');
""")

print("Setup complete!")
print(con.sql("SHOW TABLES").df())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Setup complete!
          name
0  dim_clients
1  dim_content
2   fact_daily


## 1. My rule and its reason codes

### Rule

I will prioritize content pages for refresh using historical Google Search Console performance. Pages with lower impressions, lower clicks, and poorer average search position receive higher priority because they may benefit from a content update.

### Signal Check 1 – Search Position

Reason:
Pages with worse search position generally receive less organic traffic.

Verdict: **CONFIRMED**

### Signal Check 2 – Search Impressions

Reason:
Pages with low impressions have lower visibility and may need content improvements.

Verdict: **CONFIRMED**

### Reason Codes

- REFRESH_LOW_IMPRESSIONS
- REFRESH_LOW_CLICKS
- REFRESH_POOR_POSITION

In [3]:
print("="*60)
print("Signal Check 1 : Position")
print("="*60)

position_bucket = con.sql("""
SELECT
CASE
WHEN gsc_sum_position < 10 THEN 'Top 10'
WHEN gsc_sum_position < 20 THEN '11-20'
WHEN gsc_sum_position < 50 THEN '21-50'
ELSE '50+'
END AS position_bucket,
COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks
FROM fact_daily
WHERE gsc_data_available IS TRUE
GROUP BY position_bucket
ORDER BY position_bucket;
""").df()

display(position_bucket)

print("="*60)
print("Signal Check 2 : Impressions")
print("="*60)

impression_bucket = con.sql("""
SELECT
CASE
WHEN gsc_impressions < 100 THEN 'Low'
WHEN gsc_impressions < 1000 THEN 'Medium'
ELSE 'High'
END AS impression_bucket,
COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks
FROM fact_daily
WHERE gsc_data_available IS TRUE
GROUP BY impression_bucket
ORDER BY impression_bucket;
""").df()

display(impression_bucket)

Signal Check 1 : Position


,position_bucket,n,avg_clicks
0,11-20,239116,0.015971
1,21-50,355407,0.031215
2,50+,2552939,0.314044
3,Top 10,463599,0.011182


Signal Check 2 : Impressions


,impression_bucket,n,avg_clicks
0,High,32419,5.068787
1,Low,2972453,0.057965
2,Medium,606189,0.800425


## 2. Build the ranked queue (writes the CSV)
The baseline score combines three historical signals:

- Low impressions
- Low clicks
- Poor search position

Higher scores indicate higher priority for content refresh.

Action Label:

**CONTENT_REFRESH**

Reason codes identify the main signal contributing to the recommendation.

In [4]:
from pathlib import Path

queue = con.sql("""
SELECT
content_hash_id,

gsc_impressions,

gsc_clicks,

gsc_sum_position,

(
100
- LEAST(gsc_impressions/100,40)
- LEAST(gsc_clicks/10,30)
+ LEAST(gsc_sum_position,30)
) AS baseline_score

FROM fact_daily

WHERE gsc_data_available IS TRUE
""").df()

queue["reason_code"]="REFRESH_PRIORITY"

queue["action"]="CONTENT_REFRESH"

queue=queue.sort_values(
"baseline_score",
ascending=False
)

Path("work/outputs").mkdir(
parents=True,
exist_ok=True
)

queue.to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)

display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,baseline_score,reason_code,action
56,content_76871bb7b214573c,1,0,89,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
757874,content_d76627b813646647,1,0,63,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464470,content_9ff5ae8fc64de197,1,0,66,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464466,content_26b77ed0a8bedbe3,1,0,48,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
3196575,content_a43ee3972c3179c5,1,0,34,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
174338,content_4ac3e7e9759deef4,1,0,83,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464493,content_801a7475df40e839,1,0,91,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464492,content_802b2e2fd291633a,1,0,91,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464486,content_3cce938cf330e76e,1,0,33,129.99,REFRESH_PRIORITY,CONTENT_REFRESH
1464480,content_fd161e95ff733c69,1,0,46,129.99,REFRESH_PRIORITY,CONTENT_REFRESH


## 3. Top-20 review
The top-ranked pages are reviewed manually before taking action. Higher scores indicate stronger candidates for content refresh.

Possible reasons for incorrect recommendations include seasonal traffic, newly published pages, incomplete Search Console history, or temporary ranking fluctuations.

In [5]:
top20 = queue.head(20).copy()

top20["confidence"]="Medium"

top20["why"]=[
"Low impressions and weak position"
for _ in range(len(top20))
]

top20["what_would_make_it_wrong"]=[
"Seasonality or recent publication"
for _ in range(len(top20))
]

display(top20)

,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,baseline_score,reason_code,action,confidence,why,what_would_make_it_wrong
56,content_76871bb7b214573c,1,0,89,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
757874,content_d76627b813646647,1,0,63,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464470,content_9ff5ae8fc64de197,1,0,66,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464466,content_26b77ed0a8bedbe3,1,0,48,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
3196575,content_a43ee3972c3179c5,1,0,34,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
174338,content_4ac3e7e9759deef4,1,0,83,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464493,content_801a7475df40e839,1,0,91,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464492,content_802b2e2fd291633a,1,0,91,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464486,content_3cce938cf330e76e,1,0,33,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication
1464480,content_fd161e95ff733c69,1,0,46,129.99,REFRESH_PRIORITY,CONTENT_REFRESH,Medium,Low impressions and weak position,Seasonality or recent publication


## 4. Weak picks + leakage check

Some recommendations may not require a content refresh because they are seasonal pages, recently published content, or pages with incomplete Search Console history.

Leakage Check:

- No future information was used.
- No product flags were used.
- Only historical Search Console signals available at decision time were included.

In [6]:
weak_picks = queue.tail(10)

display(weak_picks)

print("Leakage Check")

print("Future window used: NO")

print("Product flags used: NO")

print("Label-derived features used: NO")

,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,baseline_score,reason_code,action
3169375,content_eadb33b5df496f4a,34817,223,75948,67.7,REFRESH_PRIORITY,CONTENT_REFRESH
3274261,content_eadb33b5df496f4a,35404,225,77478,67.5,REFRESH_PRIORITY,CONTENT_REFRESH
461214,content_eadb33b5df496f4a,12885,226,29775,67.4,REFRESH_PRIORITY,CONTENT_REFRESH
3578638,content_eadb33b5df496f4a,34606,235,77604,66.5,REFRESH_PRIORITY,CONTENT_REFRESH
2949804,content_eadb33b5df496f4a,39305,252,86373,64.8,REFRESH_PRIORITY,CONTENT_REFRESH
1303836,content_eadb33b5df496f4a,16462,260,40170,64.0,REFRESH_PRIORITY,CONTENT_REFRESH
1653048,content_eadb33b5df496f4a,13515,264,30437,63.6,REFRESH_PRIORITY,CONTENT_REFRESH
3352538,content_e6df0936699f5b8f,14682,269,367576,63.1,REFRESH_PRIORITY,CONTENT_REFRESH
3358182,content_eadb33b5df496f4a,38436,271,84405,62.9,REFRESH_PRIORITY,CONTENT_REFRESH
1675446,content_eadb33b5df496f4a,16706,274,39378,62.6,REFRESH_PRIORITY,CONTENT_REFRESH


Leakage Check
Future window used: NO
Product flags used: NO
Label-derived features used: NO


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.